# Practice: k-nearest neighbors
*AI from scratch, model zoo 3* · lesson: https://ihor.sh/ai/zoo-knn.html

Handwritten digits: distances by hand, the nearest neighbors of one digit, choosing k on validation data, a useless
column that hijacks the distance, the curse of dimensionality measured, and kNN against logistic regression when
noise columns pile up. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it;
`hint(n)` and `answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['(Xtr - Xte[0]) has one row per training digit; square it and .sum(axis=1).', 'dist = ((Xtr - Xte[0]) ** 2).sum(axis=1)'], 2: ['np.argsort(dist) gives positions from nearest to farthest.', 'nearest = np.argsort(dist)[:5]'], 3: ['[something for k in ks], where something fits and scores one model.', 'scores = [KNeighborsClassifier(n_neighbors=k).fit(Xa, ya).score(Xv, yv) for k in ks]'], 4: ['acc_raw = KNeighborsClassifier(3).fit(Xstr, ystr).score(Xste, yste)', 'acc_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(3)).fit(Xstr, ystr).score(Xste, yste)'], 5: ['ratios = [ratio(dim) for dim in dims]'], 6: ['acc_knn = KNeighborsClassifier(3).fit(Ntr, ntr).score(Nte, nte)', 'the logistic pipeline is fitted and scored the same way']}
_ANSWERS = {1: 'ZGlzdCA9ICgoWHRyIC0gWHRlWzBdKSAqKiAyKS5zdW0oYXhpcz0xKQ==', 2: 'bmVhcmVzdCA9IG5wLmFyZ3NvcnQoZGlzdClbOjVd', 3: 'c2NvcmVzID0gW0tOZWlnaGJvcnNDbGFzc2lmaWVyKG5fbmVpZ2hib3JzPWspLmZpdChYYSwgeWEpLnNjb3JlKFh2LCB5dikgZm9yIGsgaW4ga3Nd', 4: 'YWNjX3JhdyA9IEtOZWlnaGJvcnNDbGFzc2lmaWVyKDMpLmZpdChYc3RyLCB5c3RyKS5zY29yZShYc3RlLCB5c3RlKQphY2Nfc2NhbGVkID0gbWFrZV9waXBlbGluZShTdGFuZGFyZFNjYWxlcigpLCBLTmVpZ2hib3JzQ2xhc3NpZmllcigzKSkuZml0KFhzdHIsIHlzdHIpLnNjb3JlKFhzdGUsIHlzdGUp', 5: 'cmF0aW9zID0gW3JhdGlvKGRpbSkgZm9yIGRpbSBpbiBkaW1zXQ==', 6: 'YWNjX2tubiA9IEtOZWlnaGJvcnNDbGFzc2lmaWVyKDMpLmZpdChOdHIsIG50cikuc2NvcmUoTnRlLCBudGUpCmFjY19sciA9IG1ha2VfcGlwZWxpbmUoU3RhbmRhcmRTY2FsZXIoKSwgTG9naXN0aWNSZWdyZXNzaW9uKG1heF9pdGVyPTIwMDApKS5maXQoTnRyLCBudHIpLnNjb3JlKE50ZSwgbnRlKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

d = load_digits()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.25, stratify=d.target, random_state=0)
print(Xtr.shape[0], "training digits,", Xte.shape[0], "test digits,", Xtr.shape[1], "pixels each (0 to 16)")
plt.figure(figsize=(2, 2)); plt.imshow(Xte[0].reshape(8, 8), cmap="gray_r"); plt.title(f"test digit 0: a {yte[0]}"); plt.axis("off"); plt.show()

## 1. Distances to one digit
Set `dist` to the squared distance from the first test digit `Xte[0]` to every training digit: subtract (NumPy
broadcasts the one row against all of them), square, and sum across the pixels (`axis=1`).

In [ ]:
dist = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(dist.shape if dist is not None else None, "· closest training digit is a", ytr[dist.argmin()] if dist is not None else None)

In [ ]:
check(1, lambda: dist.shape == (len(Xtr),) and ytr[dist.argmin()] == yte[0],
      "one distance per training digit (1,347 of them), and the closest one has the same label as the test digit")

## 2. The five nearest
Set `nearest` to the positions of the 5 smallest distances: `np.argsort(dist)` sorts positions by distance, and
`[:5]` keeps the first five. Then look at them.

In [ ]:
nearest = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
if nearest is not None:
    fig, ax = plt.subplots(1, 5, figsize=(6, 1.5))
    for a, i in zip(ax, nearest):
        a.imshow(Xtr[i].reshape(8, 8), cmap="gray_r"); a.set_title(ytr[i]); a.axis("off")
    plt.show()

In [ ]:
check(2, lambda: len(nearest) == 5 and dist[nearest].max() <= np.sort(dist)[4],
      "the five training digits with the smallest distances: the evidence behind the vote")

## 3. Choosing k
Hold out a validation part of the training digits (done for you). Set `scores` to the validation accuracy for each
k in `ks`: `KNeighborsClassifier(n_neighbors=k).fit(Xa, ya).score(Xv, yv)`, in a list comprehension.

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.3, stratify=ytr, random_state=1)
ks = [1, 3, 5, 9, 15, 31, 61, 121]
scores = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
if scores:
    plt.semilogx(ks, scores, "o-"); plt.xlabel("k"); plt.ylabel("validation accuracy"); plt.show()
print(scores)

In [ ]:
check(3, lambda: max(scores) > 0.97 and scores[-1] < max(scores) - 0.02,
      "small k is near 98% on these clean digits, and by k = 121 the vote smears across classes and drops")

## 4. A column that hijacks the distance
`Xs` adds one column: a scanner serial number from 0 to 1000, which says nothing about the digit. Set `acc_raw` to
a 3-neighbor kNN's test accuracy on it as is, and `acc_scaled` to the same with `StandardScaler()` first (a
pipeline, as in toolkit 2).

In [ ]:
serial = np.random.default_rng(0).uniform(0, 1000, len(d.data))
Xs = np.column_stack([d.data, serial])
Xstr, Xste, ystr, yste = train_test_split(Xs, d.target, test_size=0.25, stratify=d.target, random_state=0)
acc_raw = acc_scaled = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("raw:", acc_raw, "· scaled:", acc_scaled)

In [ ]:
check(4, lambda: acc_raw < 0.9 and acc_scaled > 0.96,
      "about 84% raw (the serial number's differences swamp the pixels') and 97% scaled")

## 5. The curse of dimensionality
`ratio(dim)` puts 1,000 random points in a cube with `dim` columns and returns the nearest distance from a random
query divided by the farthest. Set `ratios` to `ratio(dim)` for each `dim` in `dims`. Predict first: what happens to
the ratio as the columns grow?

In [ ]:
rng = np.random.default_rng(0)
def ratio(dim):
    X, q = rng.random((1000, dim)), rng.random(dim)
    dist = np.linalg.norm(X - q, axis=1)
    return dist.min() / dist.max()

dims = [2, 10, 100, 1000]
ratios = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(ratios)

In [ ]:
check(5, lambda: ratios[0] < 0.1 and ratios[-1] > 0.8,
      "about 0.02 in 2 columns and 0.9 in 1,000: in many columns everything is about equally far, so 'nearest' means little")

## 6. Noise columns: kNN against logistic regression
`with_noise(extra)` returns the digits with `extra` columns of random noise added, split. For 640 noise columns,
set `acc_knn` (3 neighbors) and `acc_lr` (`make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))`).

In [ ]:
from sklearn.linear_model import LogisticRegression
def with_noise(extra):
    noise = np.random.default_rng(0).normal(0, 16 / 3, (len(d.data), extra))
    return train_test_split(np.hstack([d.data, noise]), d.target, test_size=0.25, stratify=d.target, random_state=0)

Ntr, Nte, ntr, nte = with_noise(640)
acc_knn = acc_lr = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print("kNN:", acc_knn, "· logistic:", acc_lr)

In [ ]:
check(6, lambda: acc_knn < 0.6 and acc_lr > acc_knn + 0.25,
      "kNN about 48%, logistic about 86%: the weights learn that noise columns don't matter; a distance can't")

## 7. The price of no training
Run it: fitting kNN stores the rows, so all the work moves to prediction, and it grows with the data.

In [ ]:
for n in [1_000, 10_000, 100_000]:
    X, yy = rng.random((n, 10)), rng.integers(0, 2, n)
    m = KNeighborsClassifier(5, algorithm="brute").fit(X, yy)
    t = time.time(); m.predict(rng.random((1000, 10))); ms = (time.time() - t) * 1000
    print(f"{n:7d} stored rows: 1,000 predictions in {ms:.1f} ms")